# BE Model Behaviour

Sweeps of model parameters and burn-in to understand how each affects behavioural readouts.

**Sections:**
1. 1D parameter sweeps (psychometrics, summary stats, update matrix)
2. 2D parameter sweeps (configurable pair)
3. Burn-in effect (single param, then burn-in × η interaction)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

from Models.BE_core import BEParams, BEState, BEModel
from Data.structures import sample_stimuli, stimulus_to_category
from Analysis.summary_stats import compute_summary_stats, list_available_stats
from Analysis.update_matrix import compute_update_matrix_from_model_trace

print("Available stats:", list_available_stats())


## Config

Change values here. All sweep cells below read from this cell.

In [ ]:
# ── Simulation settings ─────────────────────────────────────────────────────
N_TRIALS  = 500
BURN_IN   = 200      # burn-in applied before each simulated session
SEED      = 42

# ── Baseline parameter values (used as fixed values when sweeping others) ───
BASE_PARAMS = dict(
    sigma_percep = 0.15,
    A_repulsion  = 0.10,
    eta_learning = 0.30,
    eta_relax    = 0.05,
)

# ── Sweep grids (1-D) ────────────────────────────────────────────────────────
SWEEP_VALUES = dict(
    sigma_percep = np.array([0.05, 0.10, 0.15, 0.25, 0.40]),
    A_repulsion  = np.array([0.00, 0.05, 0.10, 0.20, 0.35]),
    eta_learning = np.array([0.01, 0.05, 0.15, 0.30, 0.50, 0.75]),
    eta_relax    = np.array([0.001, 0.01, 0.05, 0.10, 0.20]),
)

# ── Stats to show in sweep plots ────────────────────────────────────────────
SWEEP_STATS = ['accuracy', 'recency', 'win_stay', 'stimulus_sensitivity',
               'choice_entropy', 'perseveration']

# ── 2D sweep config (change PARAM_X / PARAM_Y to switch pair) ────────────────
PARAM_X = 'eta_learning'
PARAM_Y = 'sigma_percep'
GRID_N  = 8        # points per axis  (8×8 = 64 sims)
GRID_STATS_2D = ['accuracy', 'recency', 'win_stay', 'choice_entropy']

GRID_X  = np.linspace(*BEParams.get_bounds()[PARAM_X], GRID_N)
GRID_Y  = np.linspace(*BEParams.get_bounds()[PARAM_Y], GRID_N)

# ── Burn-in sweep config ─────────────────────────────────────────────────────
BURNIN_VALUES     = [0, 50, 100, 200, 500, 1000]
BURNIN_ETA_GRID   = np.array([0.05, 0.15, 0.30, 0.50])   # for interaction plot


## Simulation Helper

In [ ]:
def simulate_session(params_dict, n_trials=N_TRIALS, burn_in=BURN_IN, seed=SEED):
    """
    Simulate one session.
    Returns dict with choices, stimuli, categories, trace.
    """
    params = BEParams(**params_dict)
    rng    = np.random.default_rng(seed)

    # Burn-in
    state = BEState.initial_uniform()
    if burn_in > 0:
        bi_stim = rng.uniform(-1, 1, burn_in)
        bi_cats = stimulus_to_category(bi_stim)
        _, _, state, _ = BEModel.simulate_session(params, state, bi_stim, bi_cats, rng)

    stim = sample_stimuli(n_trials, 'Uniform', rng)
    cats = stimulus_to_category(stim)
    choices, p_B, _, trace = BEModel.simulate_session(
        params, state, stim, cats, rng, return_history=True)

    valid = ~np.isnan(choices)
    stats = compute_summary_stats(
        choices[valid], stim[valid], cats[valid],
        stat_names=SWEEP_STATS, return_dict=True)

    return dict(params=params_dict, choices=choices, stimuli=stim,
                categories=cats, trace=trace, stats=stats, p_B=p_B)


---
## Section 1 · 1D Parameter Sweeps

For each parameter: psychometric curves, key summary stats, and update matrix.

In [ ]:
# ── Run 1D sweeps ─────────────────────────────────────────────────────────────
from Helpers.psychometry import fit_psychometric
from Helpers.utils import cumulative_gaussian

sweep_results = {}
for pname, vals in SWEEP_VALUES.items():
    sweep_results[pname] = []
    for v in vals:
        p = dict(BASE_PARAMS)
        p[pname] = float(v)
        sweep_results[pname].append(simulate_session(p))
    print(f"  {pname}: {len(vals)} points done")
print("All 1D sweeps complete.")


In [ ]:
# ── Plot: psychometric curves per param ──────────────────────────────────────
x_fine = np.linspace(-1, 1, 200)
cmap   = plt.cm.viridis

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Psychometric Curves — 1D Sweeps', fontsize=13, fontweight='bold')

for ax, pname in zip(axes.flat, SWEEP_VALUES):
    vals    = SWEEP_VALUES[pname]
    results = sweep_results[pname]
    colors  = [cmap(i / max(len(vals)-1, 1)) for i in range(len(vals))]

    for res, v, c in zip(results, vals, colors):
        valid = ~np.isnan(res['choices'])
        psych = fit_psychometric(res['stimuli'][valid], res['choices'][valid])
        if psych['success']:
            y = cumulative_gaussian(x_fine, psych['mu'], psych['sigma'],
                                    psych['lapse_low'], psych['lapse_high'])
            ax.plot(x_fine, y, color=c, lw=2, label=f'{v:.3g}')

    ax.axhline(0.5, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.axvline(0,   color='k', lw=0.5, ls='--', alpha=0.4)
    ax.set(xlabel='Stimulus', ylabel='P(B)', title=f'Sweep: {pname}',
           ylim=(-0.05, 1.05))
    ax.legend(title=pname, fontsize=7, title_fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: summary stats vs param value ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Summary Statistics — 1D Sweeps', fontsize=13, fontweight='bold')

for ax, pname in zip(axes.flat, SWEEP_VALUES):
    vals    = SWEEP_VALUES[pname]
    results = sweep_results[pname]

    for sname in SWEEP_STATS:
        sv = []
        for res in results:
            val = res['stats'].get(sname, np.nan)
            sv.append(float(val) if not isinstance(val, dict) else np.nan)
        ax.plot(vals, sv, 'o-', lw=1.5, ms=5, label=sname)

    ax.set(xlabel=pname, ylabel='Stat value', title=f'Stats vs {pname}')
    ax.axhline(0, color='k', lw=0.4, ls='--', alpha=0.4)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: update matrices per param ──────────────────────────────────────────
# Show one param at a time; pick the one with the most visible effect first.
PARAM_TO_SHOW = 'eta_learning'   # change to any param name

vals    = SWEEP_VALUES[PARAM_TO_SHOW]
results = sweep_results[PARAM_TO_SHOW]
n_vals  = len(vals)

fig, axes = plt.subplots(1, n_vals, figsize=(3*n_vals, 3.5), sharey=True)
fig.suptitle(f'Update Matrix — sweep {PARAM_TO_SHOW}  (others at baseline)',
             fontsize=12, fontweight='bold')

for ax, res, v in zip(axes, results, vals):
    um = compute_update_matrix_from_model_trace(res['trace'])
    im = ax.imshow(um, origin='lower', aspect='auto',
                   cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    ax.set_title(f'{PARAM_TO_SHOW}={v:.3g}', fontsize=9)
    ax.set_xlabel('Prev stim bin')
    if ax is axes[0]:
        ax.set_ylabel('Curr stim bin')

fig.colorbar(im, ax=axes[-1], label='ΔP(B)', shrink=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# Re-run this cell with a different PARAM_TO_SHOW to inspect other params
PARAM_TO_SHOW = 'A_repulsion'

vals    = SWEEP_VALUES[PARAM_TO_SHOW]
results = sweep_results[PARAM_TO_SHOW]
n_vals  = len(vals)

fig, axes = plt.subplots(1, n_vals, figsize=(3*n_vals, 3.5), sharey=True)
fig.suptitle(f'Update Matrix — sweep {PARAM_TO_SHOW}  (others at baseline)',
             fontsize=12, fontweight='bold')

for ax, res, v in zip(axes, results, vals):
    um = compute_update_matrix_from_model_trace(res['trace'])
    im = ax.imshow(um, origin='lower', aspect='auto',
                   cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    ax.set_title(f'{PARAM_TO_SHOW}={v:.3g}', fontsize=9)
    ax.set_xlabel('Prev stim bin')
    if ax is axes[0]:
        ax.set_ylabel('Curr stim bin')

fig.colorbar(im, ax=axes[-1], label='ΔP(B)', shrink=0.8)
plt.tight_layout()
plt.show()


---
## Section 2 · 2D Parameter Sweeps

Change `PARAM_X`, `PARAM_Y` in the Config cell and re-run this section.

All 6 pairs: `(sigma_percep, A_repulsion)`, `(sigma_percep, eta_learning)`, `(sigma_percep, eta_relax)`, `(A_repulsion, eta_learning)`, `(A_repulsion, eta_relax)`, `(eta_learning, eta_relax)`.

In [ ]:
# ── Run 2D grid simulation ────────────────────────────────────────────────────
print(f"2D sweep: {PARAM_X} × {PARAM_Y}  ({GRID_N}×{GRID_N} = {GRID_N**2} sims)")

# Pre-allocate: grid_stats[stat_name] -> (GRID_N, GRID_N) array
grid_stats_2d = {s: np.full((GRID_N, GRID_N), np.nan) for s in GRID_STATS_2D}

for i, vx in enumerate(GRID_X):
    for j, vy in enumerate(GRID_Y):
        p = dict(BASE_PARAMS)
        p[PARAM_X] = float(vx)
        p[PARAM_Y] = float(vy)
        res = simulate_session(p)
        for sname in GRID_STATS_2D:
            val = res['stats'].get(sname, np.nan)
            if not isinstance(val, dict):
                grid_stats_2d[sname][j, i] = float(val)

print("2D sweep complete.")


In [ ]:
# ── Plot: heatmaps ────────────────────────────────────────────────────────────
n_stats = len(GRID_STATS_2D)
fig, axes = plt.subplots(1, n_stats, figsize=(4.5*n_stats, 4))
fig.suptitle(f'2D sweep: {PARAM_X} (x) × {PARAM_Y} (y)', fontsize=13, fontweight='bold')

for ax, sname in zip(axes, GRID_STATS_2D):
    data = grid_stats_2d[sname]
    im = ax.imshow(data, origin='lower', aspect='auto',
                   extent=[GRID_X[0], GRID_X[-1], GRID_Y[0], GRID_Y[-1]],
                   cmap='viridis')
    ax.set(xlabel=PARAM_X, ylabel=PARAM_Y, title=sname)
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: update matrix corner cases ─────────────────────────────────────────
# Show 4 corners of the 2D grid to understand interactions
corners = [(0, 0), (0, -1), (-1, 0), (-1, -1)]
corner_labels = [
    f'{PARAM_X}={GRID_X[0]:.3g}, {PARAM_Y}={GRID_Y[0]:.3g}',
    f'{PARAM_X}={GRID_X[0]:.3g}, {PARAM_Y}={GRID_Y[-1]:.3g}',
    f'{PARAM_X}={GRID_X[-1]:.3g}, {PARAM_Y}={GRID_Y[0]:.3g}',
    f'{PARAM_X}={GRID_X[-1]:.3g}, {PARAM_Y}={GRID_Y[-1]:.3g}',
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.suptitle(f'Update matrices at 2D corners: {PARAM_X} × {PARAM_Y}',
             fontsize=11, fontweight='bold')

for ax, (ci, cj), lbl in zip(axes, corners, corner_labels):
    p = dict(BASE_PARAMS)
    p[PARAM_X] = float(GRID_X[ci])
    p[PARAM_Y] = float(GRID_Y[cj])
    res = simulate_session(p)
    um  = compute_update_matrix_from_model_trace(res['trace'])
    im  = ax.imshow(um, origin='lower', aspect='auto',
                    cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    ax.set_title(lbl, fontsize=7)
    ax.set_xlabel('Prev stim bin')
    if ax is axes[0]:
        ax.set_ylabel('Curr stim bin')

fig.colorbar(im, ax=axes[-1], label='ΔP(B)', shrink=0.8)
plt.tight_layout()
plt.show()


---
## Section 3 · Burn-in Effect

### 3a. Burn-in sweep at fixed params
How many warm-up trials are needed before single-session statistics stabilise?

In [ ]:
# ── Sweep burn-in at BASE_PARAMS ──────────────────────────────────────────────
burnin_results = []
for bi in BURNIN_VALUES:
    res = simulate_session(BASE_PARAMS, burn_in=bi)
    res['burn_in'] = bi
    burnin_results.append(res)

print("Burn-in sweep done:", BURNIN_VALUES)


In [ ]:
# ── Plot: stats vs burn-in ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(f'Effect of Burn-in  (BASE_PARAMS: η={BASE_PARAMS["eta_learning"]}, '
             f'σ={BASE_PARAMS["sigma_percep"]})', fontsize=12, fontweight='bold')

for ax, sname in zip(axes.flat, SWEEP_STATS):
    vals = [float(r['stats'].get(sname, np.nan)) for r in burnin_results]
    ax.plot(BURNIN_VALUES, vals, 'o-', lw=2, ms=6, color='steelblue')
    ax.axhline(vals[-1], color='red', lw=1, ls='--', alpha=0.5, label='asymptote')
    ax.set(xlabel='Burn-in trials', ylabel=sname, title=sname)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── Plot: update matrix vs burn-in ────────────────────────────────────────────
n = len(BURNIN_VALUES)
fig, axes = plt.subplots(1, n, figsize=(3*n, 3.5), sharey=True)
fig.suptitle('Update Matrix vs Burn-in  (BASE_PARAMS)', fontsize=11, fontweight='bold')

for ax, res in zip(axes, burnin_results):
    um = compute_update_matrix_from_model_trace(res['trace'])
    im = ax.imshow(um, origin='lower', aspect='auto',
                   cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    ax.set_title(f"burn_in={res['burn_in']}", fontsize=8)
    ax.set_xlabel('Prev stim bin')
    if ax is axes[0]:
        ax.set_ylabel('Curr stim bin')

fig.colorbar(im, ax=axes[-1], label='ΔP(B)', shrink=0.8)
plt.tight_layout()
plt.show()


### 3b. Burn-in × η interaction

Does burn-in matter more for high-η (naive) animals or low-η (expert) animals?

In [ ]:
# ── Burn-in × eta grid ────────────────────────────────────────────────────────
# For each (burn_in, eta) pair, simulate and record stats.
eta_vals = BURNIN_ETA_GRID
bi_vals  = BURNIN_VALUES

# interaction_stats[sname] -> (len(bi_vals), len(eta_vals))
interaction_stats = {s: np.full((len(bi_vals), len(eta_vals)), np.nan)
                     for s in SWEEP_STATS}

for j, eta in enumerate(eta_vals):
    for i, bi in enumerate(bi_vals):
        p = dict(BASE_PARAMS)
        p['eta_learning'] = float(eta)
        res = simulate_session(p, burn_in=bi)
        for sname in SWEEP_STATS:
            val = res['stats'].get(sname, np.nan)
            if not isinstance(val, dict):
                interaction_stats[sname][i, j] = float(val)

print("Burn-in × eta interaction complete.")


In [ ]:
# ── Plot: interaction ─────────────────────────────────────────────────────────
STAT_TO_SHOW = 'recency'   # change to any stat in SWEEP_STATS

data   = interaction_stats[STAT_TO_SHOW]
cmap_i = plt.cm.plasma
colors = [cmap_i(k / max(len(eta_vals)-1, 1)) for k in range(len(eta_vals))]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Burn-in × η interaction  [{STAT_TO_SHOW}]',
             fontsize=12, fontweight='bold')

# Left: stat vs burn_in, one line per eta
ax = axes[0]
for j, eta in enumerate(eta_vals):
    ax.plot(bi_vals, data[:, j], 'o-', color=colors[j], lw=2, ms=5,
            label=f'η={eta:.2f}')
ax.set(xlabel='Burn-in trials', ylabel=STAT_TO_SHOW,
       title='Stat vs burn-in, grouped by η')
ax.legend(fontsize=8)

# Right: heatmap
ax = axes[1]
im = ax.imshow(data, origin='lower', aspect='auto',
               extent=[eta_vals[0], eta_vals[-1], bi_vals[0], bi_vals[-1]],
               cmap='viridis')
ax.set(xlabel='η_learning', ylabel='Burn-in trials',
       title='Heatmap (colour = stat value)')
fig.colorbar(im, ax=ax, label=STAT_TO_SHOW, shrink=0.85)

plt.tight_layout()
plt.show()

# Re-run with different STAT_TO_SHOW to inspect other stats
